# Lightweight Speaker Diarization on Raspberry Pi 4 — Compression Pipeline
### Pyannote 3.1 → structured pruning → INT8 quantization → DER eval (Colab) + latency/RAM (Pi)

**Mental model — two machines, two jobs:**

| | Where | Produces | Goes into paper as |
|---|---|---|---|
| **Part A (this notebook)** | Google Colab (free GPU) | the compressed model + **DER** on AMI/VoxConverse/CALLHOME | accuracy columns |
| **Part B (`bench_rpi.py`)** | your real **Raspberry Pi 4** | **latency** + **peak RAM** | speed/memory columns |

They connect through two files this notebook writes to your Drive:
`segmentation_fp32.onnx` and `segmentation_int8.onnx`. You copy those to the Pi.

**Cost:** Colab free tier is enough (a T4 session). The only money is the Pi you already own. Everything caches to Drive so **nothing is downloaded twice**.

**Honesty notes (read these):**
- Pyannote 3.1's *actual* segmentation model may not be the "WavLM front-end" your draft assumes (recent pyannote ships a SincNet/LSTM `segmentation-3.0` and a WeSpeaker embedder, not ECAPA). This notebook **introspects the real layers** instead of assuming, so it works either way — but update the paper's architecture claims to match what `Step 5` prints.
- This is a faithful **scaffold**, not a one-click reproduction of specific numbers. Whatever DER/latency you measure here *is* your result — don't paste numbers from the draft.
- DER is computed on the **PyTorch** models. The ONNX/INT8 file is for size + the Pi's latency; `Step 12` checks FP32-vs-INT8 output agreement so you can report the quantization fidelity the paper asks for.

## Step 0 — Confirm the runtime
**Why:** make sure you actually got a GPU (Runtime → Change runtime type → T4). Pruning fine-tuning and DER eval are far faster on GPU; the rest works on CPU too.

In [ ]:
import torch, platform, subprocess
print("Python :", platform.python_version())
print("Torch  :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    print("No GPU — fine for most steps, just slower. Runtime > Change runtime type > T4.")

## Step 1 — Mount Drive and point EVERY cache at it
**Why this is the most important step for your "don't download twice" requirement:**
Colab wipes its local disk when the session ends. HuggingFace, Torch Hub, and the
corpora all default to that ephemeral disk, so you'd re-download gigabytes every time.
We redirect all of them to a single Drive folder. Set these env vars **before**
importing any library that reads them.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# One root folder on Drive holds caches, datasets, models, and results.
ROOT = '/content/drive/MyDrive/diar_rpi'
PATHS = {
    'HF':       f'{ROOT}/cache/huggingface',   # gated model weights cache here ONCE
    'TORCH':    f'{ROOT}/cache/torch',
    'DATA':     f'{ROOT}/data',                # AMI / VoxConverse / CALLHOME
    'MODELS':   f'{ROOT}/models',              # pruned checkpoints
    'EXPORT':   f'{ROOT}/export',              # .onnx files -> copy to the Pi
    'RESULTS':  f'{ROOT}/results',             # der.json etc.
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)

# Redirect caches so re-runs reuse Drive instead of re-downloading.
os.environ['HF_HOME']            = PATHS['HF']
os.environ['HUGGINGFACE_HUB_CACHE'] = f"{PATHS['HF']}/hub"
os.environ['TORCH_HOME']         = PATHS['TORCH']
os.environ['PYANNOTE_CACHE']     = f"{PATHS['HF']}/pyannote"
print("Caches now live on Drive:")
for k, v in PATHS.items():
    print(f"  {k:8s} -> {v}")

## Step 2 — Install pinned dependencies
**Why pin versions:** diarization tooling breaks across releases. These match the
ONNX Runtime 1.17.x line your paper cites. `torch-pruning` is the library that does
*real* structured channel removal (it tracks layer dependencies via a DepGraph, so
removing an output channel automatically fixes the next layer's input channels —
something `torch.nn.utils.prune` does **not** do; that one only masks weights and
gives you no size or latency win).

In [ ]:
# Quiet, and only install if missing (faster re-runs).
%pip install -q "pyannote.audio==3.1.1" "pyannote.metrics>=3.2" \
                 "torch-pruning>=1.3.0" "onnx>=1.15" "onnxruntime==1.17.1" \
                 "soundfile" "psutil" 2>/dev/null
print("Deps ready.")

## Step 3 — Authenticate with HuggingFace (gated models)
**Why:** `pyannote/speaker-diarization-3.1` and `pyannote/segmentation-3.0` are gated.
You must (1) accept the conditions on each model page in your browser once, and
(2) provide a read token here. We pull it from Colab Secrets (key icon → `HF_TOKEN`)
so it's never written in the notebook. The weights download to the Drive cache **once**.

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')      # add this in the Secrets panel
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass('HF token (hidden): ')
login(HF_TOKEN)
print("HF login OK. Accept terms at:")
print("  https://hf.co/pyannote/speaker-diarization-3.1")
print("  https://hf.co/pyannote/segmentation-3.0")

## Step 4 — Datasets, with a download guard
**Why a guard:** each corpus is large. The helper below downloads **only if the folder
is empty**, so re-running the notebook is instant after the first time.

- **AMI** — free; we fetch the headset-mix (ihm) audio + RTTM references.
- **VoxConverse** — RTTMs are on GitHub; audio is fetched by the official script (large).
- **CALLHOME** — LDC-licensed. Point `CALLHOME_DIR` at your existing copy; if it's not
  there, the notebook skips it and runs on AMI + VoxConverse so you're never blocked.

> Replace the download URLs/commands with the exact ones from each corpus's site —
> they change, and CALLHOME in particular must come from your licensed copy.

In [ ]:
import os, glob

def have_files(d, pattern='*'):
    return os.path.isdir(d) and len(glob.glob(os.path.join(d, '**', pattern), recursive=True)) > 0

DATA = PATHS['DATA']
AMI_DIR        = f'{DATA}/ami'
VOX_DIR        = f'{DATA}/voxconverse'
CALLHOME_DIR   = f'{DATA}/callhome'      # <-- point to your LDC copy if you have it
for d in (AMI_DIR, VOX_DIR, CALLHOME_DIR):
    os.makedirs(d, exist_ok=True)

# ---- AMI (example; confirm current mirror) --------------------------------
if not have_files(AMI_DIR, '*.rttm'):
    print("Downloading AMI references (run once)...")
    # RTTMs + UEMs for the diarization setup:
    !git clone -q https://github.com/pyannote/AMI-diarization-setup {AMI_DIR}/setup || true
    # Audio: use the official AMI download script for the ihm (headset-mix) condition.
    # See {AMI_DIR}/setup/pyannote/ for the file lists. (Audio is many GB.)
else:
    print("AMI already present — skipping download.")

# ---- VoxConverse ----------------------------------------------------------
if not have_files(VOX_DIR, '*.rttm'):
    print("Downloading VoxConverse v0.3 references (run once)...")
    !git clone -q https://github.com/joonson/voxconverse {VOX_DIR}/refs || true
    # Audio via the repo's download instructions (YouTube-sourced; large).
else:
    print("VoxConverse already present — skipping.")

# ---- CALLHOME (licensed) --------------------------------------------------
USE_CALLHOME = have_files(CALLHOME_DIR, '*.rttm') or have_files(CALLHOME_DIR, '*.sph')
print("CALLHOME available:", USE_CALLHOME, "(falls back to AMI+VoxConverse if False)")

### Build the evaluation file list
**Why:** all later steps iterate over a uniform list of
`(uri, audio_path, reference_rttm)` triples, so adding/removing a corpus is trivial.
Fill `build_filelist` with the actual paths once your audio is in place. Keep a small
`DEV_SUBSET` for fast smoke-tests before you commit to a full (slow) run.

In [ ]:
from pyannote.core import Annotation
from pyannote.database.util import load_rttm

def build_filelist():
    \"\"\"Return [{'uri','audio','rttm'}...]. Adapt globs to your folder layout.\"\"\"
    files = []
    # Example pattern — match RTTM stems to audio files:
    for rttm in glob.glob(f'{VOX_DIR}/**/*.rttm', recursive=True):
        uri = os.path.splitext(os.path.basename(rttm))[0]
        wav = glob.glob(f'{VOX_DIR}/**/{uri}.wav', recursive=True)
        if wav:
            files.append({'corpus': 'VoxConverse', 'uri': uri,
                          'audio': wav[0], 'rttm': rttm})
    # ... repeat for AMI and (if USE_CALLHOME) CALLHOME ...
    return files

FILES = build_filelist()
DEV_SUBSET = FILES[:5]            # 5 files for a quick smoke test
print(f"{len(FILES)} eval files found; using {len(DEV_SUBSET)} for smoke tests.")

## Step 5 — Load Pyannote 3.1 and INSPECT its real architecture
**Why inspect instead of assume:** your draft talks about a "WavLM convolutional
front-end" and "ECAPA-TDNN embeddings". The shipped 3.1 pipeline may differ. We print
the actual modules and collect the `Conv1d` layers — those are what structured pruning
targets — so the code is correct regardless of the internal architecture, and you can
**copy the printed layer names into your Methodology section** (no more guessing).

In [ ]:
from pyannote.audio import Pipeline, Model

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                    use_auth_token=HF_TOKEN)
if torch.cuda.is_available():
    pipeline.to(torch.device("cuda"))

# The segmentation model is the heavy front-end we compress:
seg_model = Model.from_pretrained("pyannote/segmentation-3.0", use_auth_token=HF_TOKEN)
seg_model.eval()

import torch.nn as nn
conv_layers = [(n, m) for n, m in seg_model.named_modules() if isinstance(m, nn.Conv1d)]
total_params = sum(p.numel() for p in seg_model.parameters())
print(f"Segmentation model: {total_params/1e6:.2f} M params, {len(conv_layers)} Conv1d layers")
for n, m in conv_layers:
    print(f"  {n:40s} in={m.in_channels:4d} out={m.out_channels:4d} k={m.kernel_size}")
# ^ Put these real numbers/names into the paper; fix the 27.4 M / front-end claims.

## Step 6 — A reusable DER evaluator (cached)
**Why cache results:** a full DER sweep is slow. We save each result to
`results/der_<tag>.json`; re-running returns the cached value instantly. DER uses a
0.25 s collar and ignores overlap — the NIST RT convention your paper states.

In [ ]:
import json
from pyannote.metrics.diarization import DiarizationErrorRate

def eval_der(pipe, files, tag, force=False):
    cache = f"{PATHS['RESULTS']}/der_{tag}.json"
    if os.path.exists(cache) and not force:
        r = json.load(open(cache)); print(f"[cached] {tag}: {r['DER']*100:.2f}%"); return r
    metric = DiarizationErrorRate(collar=0.25, skip_overlap=True)
    per_file = {}
    for f in files:
        ref = load_rttm(f['rttm'])[f['uri']]
        hyp = pipe(f['audio'])                          # run the pipeline
        per_file[f['uri']] = metric(ref, hyp, uem=None)
    r = {'tag': tag, 'DER': abs(metric), 'n_files': len(files), 'per_file': per_file}
    json.dump(r, open(cache, 'w'), indent=2)
    print(f"{tag}: {r['DER']*100:.2f}% over {len(files)} files")
    return r

## Step 7 — Baseline DER
**Why first:** every later number is reported as a delta from this. Run on `DEV_SUBSET`
to smoke-test, then on the full `FILES` for the real value. The full run is the slow one
— but it's cached, so you pay the cost once.

In [ ]:
baseline = eval_der(pipeline, DEV_SUBSET, tag="baseline_dev")   # quick check
# baseline = eval_der(pipeline, FILES, tag="baseline_full")     # real number

## Step 8 — Structured channel pruning (the real thing)
**Why torch-pruning + L1 importance:** we rank each conv layer's output channels by L1
norm and remove the lowest `ratio` fraction. The `DependencyGraph` propagates each
removal to every connected layer so the network stays valid and **actually gets smaller
and faster** (dense sub-network, no sparse masks). We prune at 0.30 and 0.59 to match
the paper's two operating points.

> If pruning the whole model drops params more than your draft's "front-end only" claim,
> that's the contradiction the reviewer flagged — restrict `ignored_layers` to keep the
> embedder/clustering untouched, OR update the paper to say you prune model-wide.

In [ ]:
import copy, torch_pruning as tp

def prune_model(base, ratio):
    model = copy.deepcopy(base).cpu().eval()
    example = torch.randn(1, 1, 16000 * 10)             # 10 s dummy waveform
    # Keep the final classification/head layer un-pruned (its out_channels = #classes).
    ignored = [m for n, m in model.named_modules()
               if isinstance(m, nn.Conv1d) and m.out_channels <= 8]
    imp = tp.importance.MagnitudeImportance(p=1)        # L1 importance
    pruner = tp.pruner.MagnitudePruner(
        model, example, importance=imp,
        pruning_ratio=ratio, ignored_layers=ignored, global_pruning=False)
    pruner.step()                                        # perform the structured removal
    n = sum(p.numel() for p in model.parameters())
    print(f"ratio={ratio:.2f} -> {n/1e6:.2f} M params")
    torch.save(model, f"{PATHS['MODELS']}/seg_pruned_{int(ratio*100)}.pt")
    return model

seg_30 = prune_model(seg_model, 0.30)
seg_59 = prune_model(seg_model, 0.59)

## Step 9 — DER of the pruned models
**Why:** quantify the accuracy cost of each sparsity level. We swap the pruned
segmentation model into a fresh pipeline and re-evaluate. (`_segmentation.model` is the
hook pyannote exposes; if the attribute name differs in your version, the inspector in
Step 5 shows the right path.)

In [ ]:
def pipeline_with(seg):
    pipe = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                    use_auth_token=HF_TOKEN)
    pipe._segmentation.model = seg.to(pipe.device if hasattr(pipe,'device') else 'cpu')
    return pipe

der_30 = eval_der(pipeline_with(seg_30), DEV_SUBSET, tag="pruned30_dev")
der_59 = eval_der(pipeline_with(seg_59), DEV_SUBSET, tag="pruned59_dev")

## Step 10 — Export to ONNX
**Why ONNX:** it's the portable format the Pi runs (via onnxruntime, no PyTorch needed),
and it's what we quantize to INT8. `opset 17` + dynamic axis on time lets the Pi feed
arbitrary-length segments. We export both the FP32 baseline (for the Pi's 1,840 ms row)
and the 59%-pruned model.

In [ ]:
def export_onnx(model, path):
    model = model.cpu().eval()
    dummy = torch.randn(1, 1, 16000 * 10)
    torch.onnx.export(
        model, dummy, path, opset_version=17,
        input_names=['waveform'], output_names=['segmentation'],
        dynamic_axes={'waveform': {2: 'samples'}, 'segmentation': {1: 'frames'}})
    print("exported", path, f"({os.path.getsize(path)/1e6:.1f} MB)")

FP32 = f"{PATHS['EXPORT']}/segmentation_fp32.onnx"
PRUNED = f"{PATHS['EXPORT']}/segmentation_pruned59.onnx"
export_onnx(seg_model, FP32)
export_onnx(seg_59, PRUNED)

## Step 11 — INT8 static quantization (per-channel)
**Why static + per-channel:** static quantization folds activation ranges in via a
calibration pass, giving the size/latency win on the Pi's integer SIMD units;
per-channel weights keep accuracy up across layers with different ranges. The
`CalibrationDataReader` feeds ~200 real 10 s segments so the ranges reflect actual audio.

In [ ]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType
import soundfile as sf, numpy as np

class AudioCalib(CalibrationDataReader):
    \"\"\"Yields real audio segments for calibration (why: synthetic noise mis-estimates
    activation ranges and hurts INT8 DER).\"\"\"
    def __init__(self, files, n=200, sr=16000, secs=10):
        self.items, self.i = [], 0
        for f in files[:n]:
            wav, _ = sf.read(f['audio'])
            wav = wav[: sr * secs]
            if len(wav) < sr * secs:                    # pad short clips
                wav = np.pad(wav, (0, sr * secs - len(wav)))
            self.items.append({'waveform': wav.reshape(1, 1, -1).astype(np.float32)})
    def get_next(self):
        if self.i >= len(self.items): return None
        x = self.items[self.i]; self.i += 1; return x

INT8 = f"{PATHS['EXPORT']}/segmentation_int8.onnx"
quantize_static(PRUNED, INT8, AudioCalib(FILES or DEV_SUBSET),
                per_channel=True, weight_type=QuantType.QInt8)
print("INT8 size:", f"{os.path.getsize(INT8)/1e6:.1f} MB  (this is your compressed model)")

## Step 12 — Verify FP32-vs-INT8 fidelity
**Why:** your paper's acceptance criterion #2 is "DER agreement ≤ 0.1 pp between FP32 and
INT8". We can't trivially run the full pipeline on ONNX, so we compare the **segmentation
logits** directly on held-out files; a tiny mean-abs-error here is strong evidence the
diarization output is unchanged. Report this number as your quantization-fidelity check.

In [ ]:
import onnxruntime as ort
def onnx_logits(path, wav):
    s = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    return s.run(None, {'waveform': wav})[0]

errs = []
for f in DEV_SUBSET:
    wav, _ = sf.read(f['audio']); wav = wav[:16000*10]
    wav = np.pad(wav, (0, 16000*10-len(wav))).reshape(1,1,-1).astype(np.float32)
    a, b = onnx_logits(PRUNED, wav), onnx_logits(INT8, wav)
    errs.append(float(np.mean(np.abs(a - b))))
print(f"FP32 vs INT8 mean-abs logit error: {np.mean(errs):.5f}  (smaller = better)")

## Step 13 — (Optional) latency-aware fine-tuning
**Why optional:** it recovers ~0.4 pp DER but costs GPU time. This is a *sketch* — wire
in pyannote's training task + AMI loader for a real run. Three short epochs at lr=1e-5 on
the pruned model is what the paper describes. Skip it for a first end-to-end pass.

In [ ]:
# Pseudocode outline — replace with pyannote.audio's Segmentation task + AMI dataloader.
# opt = torch.optim.AdamW(seg_59.parameters(), lr=1e-5)
# sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=3*len(loader))
# for epoch in range(3):
#     for batch in loader:
#         loss = seg_59.training_step(batch)   # speaker-activity loss
#         loss.backward(); opt.step(); sched.step(); opt.zero_grad()
# torch.save(seg_59, f"{PATHS['MODELS']}/seg_59_finetuned.pt")
print("Fine-tuning is optional; enable once the basic pipeline runs end-to-end.")

## Step 14 — Collect everything for the paper + the Pi
**Why:** one tidy summary on Drive, plus the two ONNX files you'll copy to the Pi.

In [ ]:
summary = {
    'segmentation_params_M': round(total_params/1e6, 2),
    'fp32_onnx_MB': round(os.path.getsize(FP32)/1e6, 1),
    'pruned59_onnx_MB': round(os.path.getsize(PRUNED)/1e6, 1),
    'int8_onnx_MB': round(os.path.getsize(INT8)/1e6, 1),
    'der_results': 'see results/der_*.json',
}
json.dump(summary, open(f"{PATHS['RESULTS']}/summary.json", 'w'), indent=2)
print(json.dumps(summary, indent=2))
print("\nNow copy these to the Pi (from Drive):")
print("  ", FP32); print("  ", INT8)

## Step 15 — Run the speed test ON THE PI
Open `bench_rpi.py` **on the Raspberry Pi 4** (not here) after copying the two ONNX files:

```bash
pip3 install onnxruntime numpy psutil
python3 bench_rpi.py --model segmentation_fp32.onnx   # the 1,840 ms baseline row
python3 bench_rpi.py --model segmentation_int8.onnx   # the compressed row
```
It prints median latency, IQR, and peak RAM — your paper's speed/memory numbers.
Copy `rpi_bench_results.json` back to Drive, then run `upload_to_github.py` to publish.

---
**Final reminder:** report only the numbers *you* measure here and on the Pi. The cloud
gives you a genuine model and genuine DER; the Pi gives you genuine latency/RAM. That's a
defensible, reproducible result — which is exactly what survives peer review.